# Sri Lanka Tea Board — annual production and exports

The Tea Board is the government authority for Ceylon tea. It publishes annual
**production** and **export volume** figures, broken down by tea type.

Connector: `ceynex/data/connectors/teaboard.py` (owner: M1 Dinapura).
Covers **2011–2025** — 15 observations.

This is the domestic authority for tea. Comtrade's tea figures are *mirror*
data, reconstructed from what importing countries reported. When the two
disagree, that disagreement is information, not an error to be smoothed away.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Is the workbook staged?

The connector reads a curated Excel workbook, resolved through the same
three-candidate search the other agriculture sources use, then the newest
`YYYY-MM-DD` snapshot inside it.

In [2]:
AGRI = cx.agriculture_raw_dir()
print("agriculture raw dir:", AGRI or "none found")
print()

workbook = cx.status(
    "Tea Board",
    (AGRI / "tea_board") if AGRI else None,
    "tea_annual_production_exports_2011_2025.xlsx",
    how="curated by M1 from Tea Board annual statistics; ask M1 for the workbook.",
)

agriculture raw dir: /ml/CeyNex/ceynex-core/data/raw

Tea Board
  staged: NO — the analysis cells below will skip.
  expected at: /ml/CeyNex/ceynex-core/data/raw/tea_board/tea_annual_production_exports_2011_2025.xlsx
  how to get it: curated by M1 from Tea Board annual statistics; ask M1 for the workbook.


## 2. What is in the workbook

Two sheets, both with the real header on **row 4** (`header=3`) — the rows above
are titles and units.

**`Production` sheet** — metric tonnes

| Column | Meaning |
|---|---|
| `orthodox_mt` | orthodox-process tea |
| `ctc_mt` | crush-tear-curl tea |
| `green_mt` | green tea |
| `total_production_mt` | all production |

**`Exports` sheet** — metric tonnes

| Column | Meaning |
|---|---|
| `bulk_mt` | unpackaged bulk tea |
| `tea_in_packets_mt` | retail packets |
| `tea_bags_mt` | tea bags |
| `instant_tea_mt` | instant tea |
| `green_tea_mt` | green tea |
| `total_exports_mt` | all exports |

Both sheets also carry `year`, `source` and `dq_flags`. The connector melts them
into long form: `year`, `metric` (`production`/`export`), `category`, `value_mt`.

## 3. Only one row per year reaches `fact_trade`

`to_fact_trade` writes **`metric == "export"` and `category == "total"`** only,
converted from tonnes to kilograms (`× 1000`).

Everything else — every production figure, and every export sub-category — is
staged but never loaded. Two separate reasons:

- **Production** has nowhere to go. `fact_trade` has no `production_volume`
  column, and writing production into `export_volume` would state something
  false: tea grown is not tea sold abroad.
- **Sub-categories** would double-count against the total if written alongside
  it, the same trap as Comtrade's World row.

So a 15-year workbook of rich detail contributes 15 rows. That is the frozen
contract being honoured rather than bent, and the agriculture agent reads the
rest from staging.

## 4. Load

In [3]:
PRODUCTION_COLUMNS = ("orthodox_mt", "ctc_mt", "green_mt", "total_production_mt")
EXPORT_COLUMNS = (
    "bulk_mt", "tea_in_packets_mt", "tea_bags_mt",
    "instant_tea_mt", "green_tea_mt", "total_exports_mt",
)

long = None
if workbook is not None:
    def read_sheet(sheet, value_columns):
        frame = pd.read_excel(workbook, sheet_name=sheet, header=3)
        frame = frame[["year", *value_columns, "source", "dq_flags"]].copy()
        frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype("int64")
        return frame

    def to_long(frame, metric, value_columns):
        out = frame.melt(
            id_vars=["year", "source", "dq_flags"],
            value_vars=list(value_columns),
            var_name="category",
            value_name="value_mt",
        ).dropna(subset=["value_mt"])
        out["category"] = (
            out["category"]
            .str.removesuffix("_production_mt")
            .str.removesuffix("_exports_mt")
            .str.removesuffix("_mt")
            .replace({"total_production": "total", "total_exports": "total"})
        )
        out.insert(1, "metric", metric)
        return out

    long = pd.concat(
        [
            to_long(read_sheet("Production", PRODUCTION_COLUMNS), "production", PRODUCTION_COLUMNS),
            to_long(read_sheet("Exports", EXPORT_COLUMNS), "export", EXPORT_COLUMNS),
        ],
        ignore_index=True,
    ).sort_values(["year", "metric", "category"], ignore_index=True)

    print(f"{len(long):,} long-form observations, {long['year'].min()}-{long['year'].max()}")
    written = long[(long["metric"] == "export") & (long["category"] == "total")]
    print(f"of which {len(written)} reach fact_trade (export totals only)")
    display(long.head())
else:
    print("skipped — no workbook staged")

skipped — no workbook staged


## 5. Production vs exports

In [4]:
if long is not None:
    totals = (
        long[long["category"] == "total"]
        .pivot_table(index="year", columns="metric", values="value_mt", aggfunc="sum")
    )
    ax = totals.plot(marker="o", title="Ceylon tea: total production vs total exports (metric tonnes)")
    ax.set_ylabel("metric tonnes")
    plt.tight_layout()
    plt.show()
    display(totals.round(0))
else:
    print("skipped — no data")

skipped — no data


## 6. Export mix — the value story behind a flat volume

Volume alone understates what matters commercially. Bulk tea leaves the country
unbranded and is repackaged elsewhere; packets and tea bags capture more of the
final price. A shift in this mix matters even when total tonnage does not move.

In [5]:
if long is not None:
    mix = (
        long[(long["metric"] == "export") & (long["category"] != "total")]
        .pivot_table(index="year", columns="category", values="value_mt", aggfunc="sum")
    )
    if not mix.empty:
        share = 100 * mix.div(mix.sum(axis=1), axis=0)
        ax = share.plot(kind="area", stacked=True, title="Tea export mix (% of exported tonnage)")
        ax.set_ylabel("% of tonnage")
        ax.set_ylim(0, 100)
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        plt.tight_layout()
        plt.show()
        display(share.round(1).tail(10))
else:
    print("skipped — no data")

skipped — no data


## 7. Cross-check against Comtrade

Two independent measurements of the same quantity. They will not match exactly,
and should not be made to.

In [6]:
if long is not None:
    facts = cx.comtrade_parquet()
    comtrade_tea = (
        facts[facts["item"] == "tea"].groupby("year")["export_volume"].sum() / 1_000_000
    ).rename("comtrade_mt_thousands")

    board_tea = (
        long[(long["metric"] == "export") & (long["category"] == "total")]
        .set_index("year")["value_mt"] / 1000
    ).rename("tea_board_mt_thousands")

    compare = pd.concat([board_tea, comtrade_tea], axis=1).dropna()
    compare["difference_pct"] = 100 * (
        compare["comtrade_mt_thousands"] - compare["tea_board_mt_thousands"]
    ) / compare["tea_board_mt_thousands"]
    display(compare.round(1))
    print()
    print("Both columns are thousands of tonnes. Differences are expected:")
    print("the Tea Board counts what left Sri Lanka; Comtrade sums what other")
    print("countries reported receiving. Re-exports, timing and under-reporting")
    print("all move the two apart. CeyNex keeps both and flags the gap.")
else:
    print("skipped — no data")

skipped — no data
